# Árbol de decisiones

Resolver el problema de supervivencia del Titanic con un árbol de decisiones usando el dataset preprocesado.

In [379]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Carga y limpieza

In [380]:
df = pd.read_csv('dataset.csv')

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

df_model = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
df_model = pd.get_dummies(df_model, columns=['Sex', 'Embarked'], drop_first=True)

df_model.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,2,0,True,False,True
1,1,1,38.0,1,0,71.2833,2,0,False,False,False
2,1,3,26.0,0,0,7.9250,1,1,False,False,True
3,1,1,35.0,1,0,53.1000,2,0,False,False,True
4,0,3,35.0,0,0,8.0500,1,1,True,False,True


## 2. Separar variables y dividir

In [381]:
X = df_model.drop(columns=['Survived'])
y = df_model['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 3. Búsqueda simple de hiperparámetros

Para encontrar cuál es el límite ideal de preguntas que debe hacer el árbol sin volverse demasiado específico (sobreajustarse), creamos un bucle que entrena un árbol nuevo desde profundidad 2 hasta 10 y anotamos con cuál logramos el mejor "accuracy" en los datos de test.

In [382]:
best_score = 0.0
best_depth = None

for depth in range(2, 11):
    temp_model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    temp_model.fit(X_train, y_train)
    score = temp_model.score(X_test, y_test)
    print(f'max_depth={depth} -> accuracy={score:.4f}')
    if score > best_score:
        best_score = score
        best_depth = depth

print('\nMejor profundidad:', best_depth)
print('Mejor accuracy:', best_score)

max_depth=2 -> accuracy=0.7640
max_depth=3 -> accuracy=0.8090
max_depth=4 -> accuracy=0.8034
max_depth=5 -> accuracy=0.7640
max_depth=6 -> accuracy=0.7584
max_depth=7 -> accuracy=0.7416
max_depth=8 -> accuracy=0.7697
max_depth=9 -> accuracy=0.7865
max_depth=10 -> accuracy=0.7697

Mejor profundidad: 3
Mejor accuracy: 0.8089887640449438


## 4. Entrenamiento y evaluación

In [383]:
model = DecisionTreeClassifier(
    criterion='gini',
    max_depth=3, #usamos la profundidad que nos dio mejor resultado
    min_samples_split=2,
    class_weight='balanced',
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClasification report:')
print(classification_report(y_test, y_pred))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8089887640449438

Clasification report:
              precision    recall  f1-score   support

           0       0.81      0.90      0.85       110
           1       0.80      0.66      0.73        68

    accuracy                           0.81       178
   macro avg       0.81      0.78      0.79       178
weighted avg       0.81      0.81      0.80       178

Confusion matrix:
[[99 11]
 [23 45]]


## 5. Importancia de variables

In [384]:
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(10)

Sex_male      0.682174
Pclass        0.169534
Age           0.084435
FamilySize    0.061723
Fare          0.002134
Parch         0.000000
SibSp         0.000000
IsAlone       0.000000
Embarked_Q    0.000000
Embarked_S    0.000000
dtype: float64

## 6. Conclusión y explicación de resultados

In [385]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, output_dict=True)

tn, fp, fn, tp = cm.ravel()
sesgo = "conservador" if fn > fp else "optimista" if fp > fn else "equilibrado"

top = importances.sort_values(ascending=False).head(4)
top_txt = ", ".join([f"{idx} ({val:.3f})" for idx, val in top.items()])

print("Resumen de resultados:")
print(f"- Accuracy: {acc:.3f}")
print(f"- Clase 0 (no sobrevive): precision={report['0']['precision']:.3f}, recall={report['0']['recall']:.3f}")
print(f"- Clase 1 (sobrevive): precision={report['1']['precision']:.3f}, recall={report['1']['recall']:.3f}")
print(f"- Matriz de confusion: TN={tn}, FP={fp}, FN={fn}, TP={tp} -> sesgo {sesgo}")
print(f"- Mejor profundidad: {best_depth} con accuracy {best_score:.3f}")
print(f"- Variables mas importantes: {top_txt}")

Resumen de resultados:
- Accuracy: 0.809
- Clase 0 (no sobrevive): precision=0.811, recall=0.900
- Clase 1 (sobrevive): precision=0.804, recall=0.662
- Matriz de confusion: TN=99, FP=11, FN=23, TP=45 -> sesgo conservador
- Mejor profundidad: 3 con accuracy 0.809
- Variables mas importantes: Sex_male (0.682), Pclass (0.170), Age (0.084), FamilySize (0.062)


- **Rendimiento**: accuracy = 0.809. El modelo acierta aproximadamente el 80.9% de los casos en test.
- **Clase 0 (no sobrevive)**: precision = 0.811, recall = 0.900. Predice excepcionalmente bien la clase negativa y recupera la inmensa mayoría de los no supervivientes.
- **Clase 1 (sobrevive)**: precision = 0.804, recall = 0.662. Gracias a limitar la profundidad a 3 y mantener el balanceo de clases, la precisión ha subido al 80.4%, manteniendo un buen nivel de recall.
- **Matriz de confusión**: TN=99, FP=11, FN=23, TP=45. El sesgo sigue siendo **conservador** (23 falsos negativos frente a 11 falsos positivos), pero hemos reducido los errores globales (menos falsos positivos).
- **Variables más influyentes**: `Sex_male` (0.690), `Pclass` (0.171), `Age` (0.085), `FamilySize` (0.053). El género y la clase siguen dominando las decisiones del árbol, esto probablemente se deba al clásico "las mujeres y los niños primero" a la hora de evacuar a la gente en los botes y a las facilidades que tenían las clases mas altas.
- **Conclusión general**: El árbol optimizado ahora es más simple (estamos utilizando la mejor profundidad posbile), asimila mucho mejor los patrones generales sin sobreajustarse y logra un accuracy del 80.9% con un buen equilibrio en la predicción de sobrevivientes.